In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [2]:

# ==========================================
# 1. CLIENT (Holds a subset of features)
# ==========================================
class VFLClient:
    def __init__(self, client_id, X_features):
        self.client_id = client_id
        self.X = X_features
        # Initialize local weights for their specific features
        self.weights = np.random.randn(X_features.shape[1]) * 0.01

    def forward_pass(self):
        """Compute partial prediction (latent representation)."""
        self.partial_pred = np.dot(self.X, self.weights)
        return self.partial_pred

    def backward_pass(self, gradient, learning_rate):
        """Update local weights based on gradient sent from server."""
        # Gradient of loss w.r.t local weights
        dw = np.dot(self.X.T, gradient) / len(self.X)
        self.weights -= learning_rate * dw



In [3]:
# ==========================================
# 2. SERVER (Holds labels and coordinates)
# ==========================================
class VFLServer:
    def __init__(self, y_true):
        self.y = y_true
        self.intercept = 0.0

    def aggregate_and_compute_loss(self, partial_preds):
        """Sum partial predictions and calculate error."""
        # The global prediction is the sum of all partial predictions + intercept
        global_pred = np.sum(partial_preds, axis=0) + self.intercept
        loss = np.mean((global_pred - self.y)**2)
        
        # Gradient of MSE loss w.r.t the global prediction
        # dL/dy_hat = 2 * (y_hat - y) / N
        gradient = 2 * (global_pred - self.y) / len(self.y)
        return global_pred, gradient, loss

    def update_intercept(self, gradient, learning_rate):
        self.intercept -= learning_rate * np.mean(gradient)



In [4]:
# ==========================================
# 3. VFL EXECUTION PIPELINE
# ==========================================
def run_vertical_fl(epochs=100, learning_rate=0.01):
    # Load Data
    housing = fetch_california_housing()
    X = StandardScaler().fit_transform(housing.data)
    y = housing.target

    # Split features vertically: Client A gets first 4 features, Client B gets last 4
    X_a = X[:, :4]
    X_b = X[:, 4:]

    client_a = VFLClient("Bank_A", X_a)
    client_b = VFLClient("RealEstate_B", X_b)
    server = VFLServer(y)

    print(f"Starting VFL Training...")
    print(f"{'Epoch':<10} | {'Global MSE':<15}")
    print("-" * 30)

    for epoch in range(epochs):
        # 1. Forward Pass: Clients compute partial predictions
        pred_a = client_a.forward_pass()
        pred_b = client_b.forward_pass()

        # 2. Server Aggregation: Combine partials and compute gradient
        global_pred, gradient, loss = server.aggregate_and_compute_loss([pred_a, pred_b])

        # 3. Backward Pass: Server sends gradient back to clients
        client_a.backward_pass(gradient, learning_rate)
        client_b.backward_pass(gradient, learning_rate)
        server.update_intercept(gradient, learning_rate)

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"{epoch:<10} | {loss:<15.4f}")

    return global_pred

if __name__ == "__main__":
    final_preds = run_vertical_fl(epochs=100, learning_rate=0.1)

Starting VFL Training...
Epoch      | Global MSE     
------------------------------
0          | 5.5901         
10         | 5.5891         
20         | 5.5881         
30         | 5.5872         
40         | 5.5862         
50         | 5.5852         
60         | 5.5843         
70         | 5.5833         
80         | 5.5824         
90         | 5.5814         
99         | 5.5805         
